In [5]:
import pyspark

In [6]:
pyspark.__file__

'/home/msimodern/de-zoomcamp2026-material/05_batch_spark/.venv/lib/python3.13/site-packages/pyspark/__init__.py'

In [7]:
from pyspark.sql import SparkSession

In [8]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/09 18:07:31 WARN Utils: Your hostname, MSI, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/09 18:07:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 18:07:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [10]:
print(spark.version)

4.1.1


In [11]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-09 18:07:47--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 108.138.137.139, 108.138.137.164, 108.138.137.70, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|108.138.137.139|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet.1’

yellow_tripdata_202 100%[===================>]  67.84M  6.10MB/s    in 8.2s    

2026-03-09 18:07:56 (8.29 MB/s) - ‘yellow_tripdata_2025-11.parquet.1’ saved [71134255/71134255]



In [12]:
df = spark.read.parquet('yellow_tripdata_2025-11.parquet')

In [13]:
df = df.repartition(4)

In [37]:
df.write.parquet('yellow/2025/11.parquet')

In [14]:
from pyspark.sql.functions import col, to_date

In [15]:
df.filter(to_date(col("tpep_pickup_datetime")) == "2025-11-15") \
.count()

162604

In [16]:
df.registerTempTable('trips_data')
spark.sql("""
select MAX(timestampdiff(HOUR, tpep_pickup_datetime, tpep_dropoff_datetime)) from trips_data
""").show()

/home/msimodern/de-zoomcamp2026-material/05_batch_spark/.venv/lib/python3.13/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)
[Stage 9:=============================>                             (2 + 2) / 4]

+---------------------------------------------------------------------+
|max(timestampdiff(HOUR, tpep_pickup_datetime, tpep_dropoff_datetime))|
+---------------------------------------------------------------------+
|                                                                   90|
+---------------------------------------------------------------------+



In [47]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-09 12:19:05--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 108.138.137.92, 108.138.137.164, 108.138.137.70, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|108.138.137.92|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv.1’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-09 12:19:05 (648 MB/s) - ‘taxi_zone_lookup.csv.1’ saved [12331/12331]



In [17]:
zone = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv.')

26/03/09 18:09:19 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: taxi_zone_lookup.csv..
java.io.FileNotFoundException: File taxi_zone_lookup.csv. does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.Res

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/msimodern/de-zoomcamp2026-material/05_batch_spark/taxi_zone_lookup.csv.. SQLSTATE: 42K03

In [4]:
from pyspark.sql.functions import col
zone = zone.withColumn("LocationID", col("LocationID").cast("int"))

zone.registerTempTable('zones')

spark.sql("""
select t2.Zone, count(*) as num_trips from trips_data t1 inner join zones t2 on t1.PULocationID = t2.LocationID
group by t2.Zone
order by num_trips
LIMIT 1
""").show(truncate=False)

NameError: name 'zone' is not defined